# Advanced Retrieval with the Cat Health PDF

Session 1 used a dense vector retriever:

```text
question -> embed -> nearest chunks -> answer
```

This notebook keeps the same cat-health PDF and compares dense vector search, BM25, parent-child retrieval, hybrid retrieval, Cohere reranking, and multi-query retrieval.

> This is a retrieval exercise, not veterinary guidance. Answer only from retrieved context. Recommend a veterinarian for diagnosis, treatment, medication, or urgent-care decisions.


## Learning Outcomes

By the end of this session, you will be able to:

- Explain the different failure modes of dense and BM25 retrieval.
- Fuse independent ranked lists with reciprocal rank fusion (RRF).
- Increase recall with multiple generated search queries.
- Search focused child chunks while returning useful parent-page context.
- Use Cohere Rerank as a second-stage ranker.
- Compare retrieval systems with reviewed cases, visible evidence, metrics, and latency.


## Build Order

Build and compare the following layers:

1. Naive dense RAG: in-memory Qdrant, OpenAI embeddings, nearest child chunks, and a grounded answer.
2. BM25: sparse lexical retrieval over the same chunks.
3. Parent-child retrieval: search precise child chunks and return their parent PDF pages.
4. Hybrid retrieval with Cohere reranking: fuse dense and BM25 candidates with reciprocal rank fusion (RRF), recover parent pages, then rerank them.
5. Multi-query retrieval: generate alternate searches before the hybrid retrieve-then-rerank pipeline.

Dense retrieval plus BM25 is **hybrid retrieval** (or hybrid search). RRF is an **ensemble** method that combines their ranked lists. Adding Cohere reranking creates a **two-stage hybrid retrieve-then-rerank pipeline**.

Extra stages add latency, cost, and sometimes noise. Evaluate each added stage against those trade-offs.


## Task 1: Setup

Install the project environment from the session folder with `uv sync`, then run this notebook from that same folder. You need `OPENAI_API_KEY` and `COHERE_API_KEY` available when the notebook prompts for them.


In [2]:
from __future__ import annotations

import os
import re
from dataclasses import replace
from getpass import getpass
from pathlib import Path
from typing import Iterable, Sequence

import pandas as pd
from langchain_cohere import CohereRerank
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_qdrant import QdrantVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from rank_bm25 import BM25Okapi

from lib import (
    AnswerOutput,
    EvalCase,
    EvalItem,
    RetrievedDocument,
    answer_similarity_scorer,
    compare_eval_reports,
    compare_reports,
    faithfulness_scorer,
    make_openai_faithfulness_judge,
    run_eval,
    run_retrieval_eval,
)


/var/folders/nn/sjkkz9tj6nn1g3sq5w_dm_zw0000gn/T/ipykernel_26229/2795770623.py:12: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [3]:
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API Key: ")

if not os.environ.get("COHERE_API_KEY"):
    os.environ["COHERE_API_KEY"] = getpass("Cohere API Key: ")

CHAT_MODEL = os.environ.get("AIM_CHAT_MODEL", "gpt-5.4-mini")
EVAL_MODEL = os.environ.get("AIM_EVAL_MODEL", CHAT_MODEL)
EMBEDDING_MODEL = os.environ.get("AIM_EMBEDDING_MODEL", "text-embedding-3-small")
RERANK_MODEL = os.environ.get("AIM_RERANK_MODEL", "rerank-v4.0-fast")

embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)
answer_model = ChatOpenAI(model=CHAT_MODEL, temperature=0)
evaluation_model = ChatOpenAI(model=EVAL_MODEL, temperature=0)
query_model = ChatOpenAI(model=CHAT_MODEL, temperature=0)

print(f"Chat model: {CHAT_MODEL}")
print(f"Evaluation model: {EVAL_MODEL}")
print(f"Embedding model: {EMBEDDING_MODEL}")
print(f"Cohere rerank model: {RERANK_MODEL}")


Chat model: gpt-5.4-mini
Evaluation model: gpt-5.4-mini
Embedding model: text-embedding-3-small
Cohere rerank model: rerank-v4.0-fast


## Task 2: Load and Chunk the Cat PDF for Naive RAG

Load the bundled PDF as page-level documents, then split each page into focused chunks for the dense RAG baseline. Preserve source and page metadata on every chunk.


In [4]:
pdf_path = Path("data/cat_health_guidelines.pdf")
if not pdf_path.exists():
    raise FileNotFoundError(f"Expected the cat PDF at {pdf_path.resolve()}")

pages = [page for page in PyPDFLoader(str(pdf_path)).load() if page.page_content.strip()]
for page_number, page in enumerate(pages, start=1):
    page.metadata.update(
        {
            "source": pdf_path.name,
            "page": page_number,
            "page_id": f"page-{page_number:02d}",
            "document_type": "cat_health_guideline",
        }
    )

print(f"Loaded {len(pages)} text-containing PDF pages.")
print(pages[0].metadata)


Loaded 22 text-containing PDF pages.
{'producer': 'Acrobat Distiller 10.0.0 (Windows)', 'creator': 'Adobe InDesign CS6 (Windows)', 'creationdate': '2021-02-02T08:52:15-05:00', 'author': '7123', 'moddate': '2021-02-02T07:53:51-07:00', 'title': 'djs_jaaha_56_5_COVER.indd', 'source': 'cat_health_guidelines.pdf', 'total_pages': 22, 'page': 1, 'page_label': '1', 'page_id': 'page-01', 'document_type': 'cat_health_guideline'}


In [5]:
child_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=120,
    add_start_index=True,
)
children = child_splitter.split_documents(pages)
for index, child in enumerate(children):
    child.metadata["chunk_id"] = f"child-{index:03d}"

print(f"Created {len(children)} child chunks from {len(pages)} parent pages.")
print(children[0].metadata)
print(children[0].page_content[:500])


Created 159 child chunks from 22 parent pages.
{'producer': 'Acrobat Distiller 10.0.0 (Windows)', 'creator': 'Adobe InDesign CS6 (Windows)', 'creationdate': '2021-02-02T08:52:15-05:00', 'author': '7123', 'moddate': '2021-02-02T07:53:51-07:00', 'title': 'djs_jaaha_56_5_COVER.indd', 'source': 'cat_health_guidelines.pdf', 'total_pages': 22, 'page': 1, 'page_label': '1', 'page_id': 'page-01', 'document_type': 'cat_health_guideline', 'start_index': 0, 'chunk_id': 'child-000'}
VETERINARY PRACTICE GUIDELINES
2021 AAHA/AAFP Feline Life Stage Guidelines*
Jessica Quimby, DVM, PhD, DACVIM y, Shannon Gowland, DVM, DABVP y, Hazel C. Carney, DVM, MS, DABVP,
Theresa DePorter, DVM, MRCVS, DACVB, DECAWBM, Paula Plummer, LVT, VTS (ECC, SAIM), Jodi Westropp,
DVM, PhD, DACVIM
ABSTRACT
The guidelines, authored by a Task Force ofexperts in feline clinical medicine, are an update and extension of the AAFP–AAHA
Feline Life Stage Guidelines published in 2010. The guidelines are publishe


### ❓ Question 1: Traceability

Why should every child chunk retain its source file and page metadata?

Every child shuold keep source file and page metadata so retrieved evidence can be traceable to the original PDF for inspection, parent pages can be recovered from child hits, answers can cite sources, and retriever approaches can be evalulated against the same page level ground truth. 


### Shared Result Representation

All retrievers return the same RetrievedDocument type. Stable chunk and page IDs make the evidence inspectable and keep later comparisons fair.


In [6]:
def as_retrieved_document(document: Document, score: float | None = None) -> RetrievedDocument:
    return RetrievedDocument(
        id=document.metadata["chunk_id"],
        text=document.page_content,
        score=float(score) if score is not None else None,
        evidence_ids=(document.metadata["page_id"],),
        metadata=dict(document.metadata),
    )


def print_results(results: Sequence[RetrievedDocument], text_limit: int = 260) -> None:
    for rank, result in enumerate(results, start=1):
        page = result.metadata.get("page", "?")
        score = "n/a" if result.score is None else f"{result.score:.4f}"
        print(f"#{rank} | {result.id} | page={page} | score={score}")
        print(result.text[:text_limit].replace("\n", " "))
        print()


## Task 3: Naive Dense Vector RAG with In-Memory Qdrant

Session 1 baseline:

question -> OpenAI embedding -> Qdrant nearest-neighbor search -> answer

Create an in-memory Qdrant collection from the child chunks. Retrieve the nearest chunks, then pass only those chunks to the answer model. Use this **naive RAG baseline** for later comparisons.


In [6]:
# Qdrant stays in memory for this notebook. It disappears when the kernel stops.
vector_store = QdrantVectorStore.from_documents(
    documents=children,
    embedding=embeddings,
    location=":memory:",
    collection_name="cat_health_naive_dense_rag",
)

FIRST_STAGE_K = 8


def dense_retrieve(question: str, k: int = 5) -> list[RetrievedDocument]:
    matches = vector_store.similarity_search_with_score(question, k=k)
    return [as_retrieved_document(document, score) for document, score in matches]


dense_preview = dense_retrieve("What should a senior cat wellness visit cover?", k=4)
print_results(dense_preview)


#1 | child-034 | page=7 | score=0.6789
disease. Mature Adult and Senior Cats The medical history and examination of mature adult and senior cats will be focused on early detection of disease. Adult and senior cats are often diagnosed with comorbidities. Speci ﬁc questions regarding changes in appet

#2 | child-021 | page=6 | score=0.6741
For example, some senior cats aged 10 years and older may remain in excellent physical condition and would be best treated as a mature adult at the veterinarian ’s discretion. The guidelines are intended to be a starting point from which individualized care re

#3 | child-060 | page=10 | score=0.6188
study, three 10- to 15-minute exercise sessions per day led to a loss of approximately 1% of body weight in 1 month with no food intake restrictions. 66 Senior Cats Senior cats exhibiting new or unusual behavior should be evaluated for medical conditions. 12 C

#4 | child-036 | page=8 | score=0.6182
Detecting signs of pain or anxiety and evaluation of qual

### Naive RAG Answer

Retrieval quality and answer quality are separate concerns. Pass the nearest dense chunks to the answer model. The same grounded-answer function will later receive context from the other retrievers.


In [7]:
answer_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "Answer only from the provided cat-health guideline context. "
            "Do not diagnose, prescribe, or make an urgent-care decision. "
            "If the context is insufficient, say so. End with a short Sources line.",
        ),
        ("human", "Question: {question}\n\nContext:\n{context}"),
    ]
)


def format_context(documents: Sequence[RetrievedDocument]) -> str:
    blocks = []
    for document in documents:
        page = document.metadata.get("page", "unknown")
        blocks.append(f"[source={document.id}; page={page}]\n{document.text}")
    return "\n\n".join(blocks)


def answer_with_retriever(
    question: str,
    retriever,
    k: int = 4,
) -> AnswerOutput:
    documents = retriever(question, k=k)
    response = answer_model.invoke(
        answer_prompt.format_messages(question=question, context=format_context(documents))
    )
    return AnswerOutput(answer=str(response.content), documents=tuple(documents))


naive_result = answer_with_retriever(
    "What should an owner discuss at a senior cat wellness visit?",
    retriever=dense_retrieve,
)
print(naive_result.answer)
print("\nNaive dense evidence:")
print_results(naive_result.documents, text_limit=200)


At a senior cat wellness visit, an owner should discuss:

- Changes in appetite
- Increased drinking and urination
- Vomiting, hairball vomiting, or diarrhea
- Increased nighttime activity or vocalization
- Changes in the cat’s normal habits or activity
- Changes in litter box use
- Any new or unusual behavior

These topics can help the veterinarian look for issues such as cognitive changes, reduced mobility, pain, reduced vision or hearing, and other disease-related changes.

Sources: child-034 p.7; child-060 p.10; child-021 p.6

Naive dense evidence:
#1 | child-034 | page=7 | score=0.6957
disease. Mature Adult and Senior Cats The medical history and examination of mature adult and senior cats will be focused on early detection of disease. Adult and senior cats are often diagnosed with 

#2 | child-021 | page=6 | score=0.6693
For example, some senior cats aged 10 years and older may remain in excellent physical condition and would be best treated as a mature adult at the veterinarian 

## Task 4: BM25 Sparse Retrieval

BM25 ranks the same child chunks with lexical term matches rather than embeddings. It is useful for abbreviations, age ranges, named conditions, and phrases that dense similarity may blur.

Keep the corpus, chunks, and retrieval depth fixed. The retriever is the only thing that changes.


In [8]:
def tokenize(text: str) -> list[str]:
    return re.findall(r"[a-z0-9]+", text.lower())


bm25 = BM25Okapi([tokenize(child.page_content) for child in children])


def bm25_retrieve(question: str, k: int = 5) -> list[RetrievedDocument]:
    scores = bm25.get_scores(tokenize(question))
    ranked_indices = sorted(range(len(scores)), key=lambda index: scores[index], reverse=True)
    return [
        as_retrieved_document(children[index], float(scores[index]))
        for index in ranked_indices[:k]
    ]


bm25_preview = bm25_retrieve("What do BCS and MCS stand for?", k=4)
print_results(bm25_preview)


#1 | child-079 | page=13 | score=11.1440
could result in health problems. 83 No matter the life stage, to help avoid potential nutrient insufﬁciencies, cats should be fed diets labeled with an Association of American Feed Control Of ﬁcials statement of nutritional adequacy. AAHA and the AAFP do not a

#2 | child-029 | page=6 | score=10.8317
Evaluation and recording of body weight, body condition score (BCS), and muscle condition score (MCS) are important compo- nents of the physical examination at all life stages to allow early 56 JAAHA | 57:2 Mar/Apr 2021

#3 | child-006 | page=1 | score=9.7786
BCS (body condition score); DER (daily energy requirements); DJD (degenerative joint disease); FCV (feline calicivirus); FeLV (feline leukemia virus); FHV-1 (feline herpesvirus type 1); FIC (feline idiopathic cystitis); FPV (feline panleukopenia virus); GI (ga

#4 | child-082 | page=13 | score=9.7514
vidual patient. Recommendations can be found in the AAFP ’s Feline Feeding Programs Consensus S

### ❓ Question 2: When Should BM25 Win?

Which query is more likely to favor BM25: `What do BCS and MCS stand for?` or `How should an owner get ready for a senior-cat wellness visit?` Why?

"What do BCS and MCS stand for?' acronym question would do better with BM25 because BM25 is lexical. This approach ranks chunks by how well the query's exact tokens match the chunk text. 

Therefore BM25 is more useful for exact terms, acronyms, abbreviations, named terms, numbers than dense similarity. BM25 gets a direct hit, dense can miss the best chunk by putting generic chunks first as dense retrieval looks for semantic similarity - BCS and MCS could be semantically close to many guideline text and could return back irrelevant chunks. 

The senior wellness query favors dense retrieval because this question is a paraphrase/conceptual question. Dense embedding is better at matching meaning across different phrasings. BM25 is looking for word overlap - and in the PDF it could say 'discuss', 'prepare' not necessarily 'get ready' and BM25 may miss that. 



### 🚀 Activity 1: Dense vs. BM25 Failure Modes

Pick three questions: one paraphrase-heavy question, one exact-term question, and one broad multi-part question. Inspect both result lists before generating an answer.

- Which retriever put the best evidence first?
- Which retriever returned redundant chunks?
- Which question type should favor BM25, and why?


In [9]:
activity_questions = [
    "What does the guideline say about BCS and MCS?",
    "How often should senior cats see a veterinarian?",
    "What factors shape an individualized preventive-care plan?",
]

for question in activity_questions:
    print(f"\nQUESTION: {question}\n")
    print("DENSE")
    print_results(dense_retrieve(question, k=3), text_limit=150)
    print("BM25")
    print_results(bm25_retrieve(question, k=3), text_limit=150)



QUESTION: What does the guideline say about BCS and MCS?

DENSE
#1 | child-005 | page=1 | score=0.4263
mendations should not be construed as dictating an exclusive protocol, course of treatment, or procedure. Variations in practice may be war- ranted ba

#2 | child-029 | page=6 | score=0.4119
Evaluation and recording of body weight, body condition score (BCS), and muscle condition score (MCS) are important compo- nents of the physical exami

#3 | child-006 | page=1 | score=0.3845
BCS (body condition score); DER (daily energy requirements); DJD (degenerative joint disease); FCV (feline calicivirus); FeLV (feline leukemia virus);

BM25
#1 | child-029 | page=6 | score=12.3015
Evaluation and recording of body weight, body condition score (BCS), and muscle condition score (MCS) are important compo- nents of the physical exami

#2 | child-082 | page=13 | score=11.5603
vidual patient. Recommendations can be found in the AAFP ’s Feline Feeding Programs Consensus Statement. 19 Young Adult Cats

In [19]:
activity_questions = [
    "What does the guideline say about BCS and MCS?",
    "How often should senior cats see a veterinarian?",
    "What factors shape an individualized preventive-care plan?",
]

for question in activity_questions:
    print(f"\n{'='*60}")
    print(f"QUESTION: {question}\n")

    dense_result = answer_with_retriever(question, retriever=dense_retrieve, k=3)
    print("DENSE ANSWER:")
    print(dense_result.answer)
    print("\nDense evidence:")
    print_results(dense_result.documents, text_limit=150)

    bm25_result = answer_with_retriever(question, retriever=bm25_retrieve, k=3)
    print("\nBM25 ANSWER:")
    print(bm25_result.answer)
    print("\nBM25 evidence:")
    print_results(bm25_result.documents, text_limit=150)


QUESTION: What does the guideline say about BCS and MCS?

DENSE ANSWER:
The guideline says that body condition score (BCS) and muscle condition score (MCS) should be evaluated and recorded along with body weight as part of the physical examination at all life stages, because this helps allow early assessment/identification of changes.

Sources: child-029 p.6; child-006 p.1

Dense evidence:
#1 | child-005 | page=1 | score=0.4263
mendations should not be construed as dictating an exclusive protocol, course of treatment, or procedure. Variations in practice may be war- ranted ba

#2 | child-029 | page=6 | score=0.4119
Evaluation and recording of body weight, body condition score (BCS), and muscle condition score (MCS) are important compo- nents of the physical exami

#3 | child-006 | page=1 | score=0.3845
BCS (body condition score); DER (daily energy requirements); DJD (degenerative joint disease); FCV (feline calicivirus); FeLV (feline leukemia virus);


BM25 ANSWER:
The guideline says 

# Activity #1 Notes 

- Which retriever put the best evidence first?

Question 1: What does the guideline say about BCS and MCS?  BM25 puts the best evidence first 
Dense: child-005/page 1. Generic. 
BM25: child-0029/page 6. Good, on topic 

Question 2: How often should senior cats see a veterinarian? Dense semantic matching ranked senior cat care first 

Dense: child-021/page 6. Good, on topic 
BM25: child-072/page 12. No, off topic 

Question 3: What factors shape an individualized preventive-care plan? Both ranked same chunk first, and on tpoic. If taking latency into account, BM25 likely wins

Dense: child-023/page 6. Good, on topic 
BM25: child-023/page 6. Good, on topic 

- Which retriever returned redundant chunks?

BM25 shows more redundancy
Q1: child-029 and child-025 are both from page 6
Q2 + Q1: child-082 (page 13) 
Q3: child-023 and child-025 are both from page 6

BM25 returns several chunks from the same page, likely because they share same query term overlaps e.g., BCS, MCS, cat 

Dense can also be redundant but its more spread out. It returns redundant pages, but different chunks from that page. 023, 021, 029

- Which question type should favor BM25, and why?

Questions with exact terms/acroynyms like What does the guideline say about BCS and MCS?

BM25 is better at literal token matches.

- Generated answers 

Both retrievers produced usable answers for all three questions, but the specifics and nuanced differed a bit reflecting different retrieval quality and ranking. The final generated answers were probably good because k > 1 and LLM synthesis was able to work with the chunks retrieved from both. This may have worked because examples were simple, but for more hard, complex edge cases, we probably would have noticed the impact of the different retrieval approaches 

## Task 5: Parent-Child Retrieval

Build a parent-child pipeline on top of the dense retriever:

question -> dense child search in Qdrant -> matching parent PDF pages

Child chunks give the vector store a focused search surface. Parent pages give the answer model surrounding context. Parent-child retrieval adds context recovery to dense retrieval; it is not hybrid retrieval.


In [10]:
# Parent-page lookup begins here; naive RAG does not need parent records.
parents_by_id = {page.metadata["page_id"]: page for page in pages}


def recover_parent_documents(
    child_candidates: Sequence[RetrievedDocument],
    k: int,
) -> list[RetrievedDocument]:
    parents: list[RetrievedDocument] = []
    seen_parent_ids: set[str] = set()

    for child in child_candidates:
        page_id = child.metadata["page_id"]
        if page_id in seen_parent_ids:
            continue
        parent = parents_by_id[page_id]
        parents.append(
            RetrievedDocument(
                id=page_id,
                text=parent.page_content,
                score=child.score,
                evidence_ids=(page_id,),
                metadata={
                    **parent.metadata,
                    "retrieved_from_child": child.id,
                    "first_stage_score": child.score,
                },
            )
        )
        seen_parent_ids.add(page_id)
        if len(parents) == k:
            break
    return parents


def parent_child_dense_retrieve(question: str, k: int = 5) -> list[RetrievedDocument]:
    child_candidates = dense_retrieve(question, k=FIRST_STAGE_K)
    return recover_parent_documents(child_candidates, k=k)


parent_preview = parent_child_dense_retrieve(
    "How should an owner prepare for a senior cat wellness visit?",
    k=3,
)
print_results(parent_preview, text_limit=400)


#1 | page-06 | page=6 | score=0.6861
For example, some senior cats aged 10 years and older may remain in excellent physical condition and would be best treated as a mature adult at the veterinarian ’s discretion. The guidelines are intended to be a starting point from which individualized care recommenda- tions can be developed. Discussion Items for All Life Stages The Task Force recommends a minimum of annual examinations for all ca

#2 | page-07 | page=7 | score=0.6555
detection of changes and identi ﬁcation of trends. 20 Obtaining dorsal and lateral photographs of the patient is recommended to facilitate monitoring BCS/MCS as the cat ages, and can help the owner recognize subtle changes. Diseases and conditions that require additional focus during the examination by each life stage are listed in Table 3. Kittens Kittens will have different health risks dependin

#3 | page-10 | page=10 | score=0.6434
including carpeting, window and door frames, curtains, and couches. Keeping the nail

### ❓ Question 3: Search Small, Return Large?

Why does parent-child retrieval search focused child chunks but return the larger parent pages instead of indexing and returning only full pages?

Parent-child retrieval splits the job into 2 steps because the search and generation need different chunk sizes. Child chunks give the vector store a focused search surface. Parent pages give the answer model surrounding context. 

This is because if we embed and retrieve whole PDF pages, each vector represents many topics at once, thereby making the page level embedding is diluted by everything on that page. Dense search becomes noisier and less precise.  

Therefore smaller child chunks fix that because each embedding targets one focused idea, best matching unit is easier to find (in this lab we created 159 child chunks from 22 pages) - so searching for children makes it a better index to find the right place in the document. 

But the problem with returning only child chunks (naive RAG) is that child text is often truncated mid thought, missing nearby context, too narrow for the LLM to synthesize a complete answer - and parent child retrieval fixes that by using the child hit only as a pointer 



# Breakout Room #2: Combine, Rank, and Expand

The first half produced three building blocks: dense retrieval, BM25, and parent-child context recovery. This breakout combines them and compares the results.

In this breakout:

1. Fuse dense and BM25 rankings with reciprocal rank fusion.
2. Rerank the broad hybrid candidate set with Cohere.
3. Expand vague questions into multiple searches.
4. Compare the trade-offs with the local evaluation library.

Keep only the stages that improve measured retrieval enough to justify their latency.


## Task 6: Hybrid Retrieval — Dense + BM25

Dense and BM25 scores use different scales, so raw scores cannot be added directly. Reciprocal rank fusion (RRF) works from their **ranked lists** instead.

The hybrid first stage is:

dense candidates + BM25 candidates -> RRF -> hybrid child candidates

**Hybrid retrieval** combines semantic vector retrieval with sparse lexical retrieval. RRF makes it an **ensemble retriever** by fusing multiple rankings.


In [11]:
def reciprocal_rank_fusion(
    ranked_lists: Iterable[Sequence[RetrievedDocument]],
    *,
    limit: int,
    rrf_constant: int = 60,
) -> list[RetrievedDocument]:
    scores: dict[str, float] = {}
    documents_by_id: dict[str, RetrievedDocument] = {}

    for ranked_list in ranked_lists:
        for rank, document in enumerate(ranked_list, start=1):
            documents_by_id.setdefault(document.id, document)
            scores[document.id] = scores.get(document.id, 0.0) + 1 / (rrf_constant + rank)

    return [
        replace(
            documents_by_id[document_id],
            score=score,
            metadata={
                **documents_by_id[document_id].metadata,
                "rrf_score": score,
            },
        )
        for document_id, score in sorted(scores.items(), key=lambda item: item[1], reverse=True)[:limit]
    ]


def hybrid_children_retrieve(question: str, k: int = 5) -> list[RetrievedDocument]:
    return reciprocal_rank_fusion(
        [
            dense_retrieve(question, k=FIRST_STAGE_K),
            bm25_retrieve(question, k=FIRST_STAGE_K),
        ],
        limit=k,
    )


hybrid_preview = hybrid_children_retrieve("What does the guideline say about BCS and MCS?", k=4)
print_results(hybrid_preview)


#1 | child-029 | page=6 | score=0.0325
Evaluation and recording of body weight, body condition score (BCS), and muscle condition score (MCS) are important compo- nents of the physical examination at all life stages to allow early 56 JAAHA | 57:2 Mar/Apr 2021

#2 | child-030 | page=7 | score=0.0310
detection of changes and identi ﬁcation of trends. 20 Obtaining dorsal and lateral photographs of the patient is recommended to facilitate monitoring BCS/MCS as the cat ages, and can help the owner recognize subtle changes. Diseases and conditions that require

#3 | child-005 | page=1 | score=0.0164
mendations should not be construed as dictating an exclusive protocol, course of treatment, or procedure. Variations in practice may be war- ranted based on the needs of the individual patient, resources, and limitations unique to each individual practice sett

#4 | child-082 | page=13 | score=0.0161
vidual patient. Recommendations can be found in the AAFP ’s Feline Feeding Programs Consensus Stat

## Task 7: Cohere Reranking over Hybrid Candidates

Use hybrid retrieval to gather a broad set of plausible child chunks from dense and BM25 search. Recover their parent pages, then send those candidates and the question to Cohere Rerank.

dense + BM25 -> RRF -> parent pages -> Cohere Rerank -> final context

The result is a **two-stage hybrid retrieve-then-rerank pipeline**. Cohere scores rank documents within one query and candidate set; do not treat them as universal probabilities.

The custom RRF function supplies the candidate list, so the notebook calls LangChain's `CohereRerank` compressor directly. A single LangChain `BaseRetriever` could instead be wrapped with `ContextualCompressionRetriever`.


In [12]:
RERANK_CANDIDATE_K = 8


def to_langchain_document(candidate: RetrievedDocument) -> Document:
    return Document(
        page_content=candidate.text,
        metadata={
            **candidate.metadata,
            "retrieved_id": candidate.id,
            "evidence_ids": list(candidate.canonical_evidence_ids),
            "first_stage_score": candidate.score,
        },
    )


def to_reranked_document(document: Document) -> RetrievedDocument:
    relevance_score = document.metadata.get("relevance_score")
    return RetrievedDocument(
        id=document.metadata["retrieved_id"],
        text=document.page_content,
        score=float(relevance_score) if relevance_score is not None else None,
        evidence_ids=tuple(document.metadata["evidence_ids"]),
        metadata=dict(document.metadata),
    )


def rerank_parent_candidates(
    question: str, candidates: Sequence[RetrievedDocument], k: int
) -> list[RetrievedDocument]:
    compressor = CohereRerank(model=RERANK_MODEL, top_n=k)
    reranked_documents = compressor.compress_documents(
        documents=[to_langchain_document(candidate) for candidate in candidates],
        query=question,
    )
    return [to_reranked_document(document) for document in reranked_documents]


def hybrid_reranked_retrieve(question: str, k: int = 5) -> list[RetrievedDocument]:
    child_candidates = hybrid_children_retrieve(question, k=RERANK_CANDIDATE_K)
    parent_candidates = recover_parent_documents(child_candidates, k=RERANK_CANDIDATE_K)
    return rerank_parent_candidates(question, parent_candidates, k)


rerank_preview = hybrid_reranked_retrieve(
    "How should an owner prepare for a senior cat wellness visit?",
    k=3,
)
print_results(rerank_preview, text_limit=400)


#1 | page-06 | page=6 | score=0.7716
For example, some senior cats aged 10 years and older may remain in excellent physical condition and would be best treated as a mature adult at the veterinarian ’s discretion. The guidelines are intended to be a starting point from which individualized care recommenda- tions can be developed. Discussion Items for All Life Stages The Task Force recommends a minimum of annual examinations for all ca

#2 | page-18 | page=18 | score=0.7030
events to increase knowledge and con ﬁdence when taking patient histories (see “Conducting Effective Patient Histories” box) and pro- viding client education are just as important as further education on feline-friendly handling, disease processes, and technical skills. Ideally, client education is a key responsibility for all staff members. Every life stage will have speci ﬁc items that should be

#3 | page-10 | page=10 | score=0.6923
including carpeting, window and door frames, curtains, and couches. Keeping the nai

## Task 8: Multi-Query Retrieval

A user question may be vague, incomplete, or phrased differently from the source. Generate alternate search queries, run each through hybrid first-stage retrieval, and fuse the resulting child rankings.

Multi-query retrieval expands recall on top of the hybrid retrieve-then-rerank pipeline. It also adds model calls, latency, and possible noise.


In [13]:
multi_query_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "Write {count} short, distinct search queries for the cat health guideline PDF. "
            "Return one query per line. Do not answer the question.",
        ),
        ("human", "{question}"),
    ]
)


def generate_query_variants(question: str, count: int = 3) -> list[str]:
    response = query_model.invoke(
        multi_query_prompt.format_messages(question=question, count=count)
    )
    variants = [question]
    for line in response.content.splitlines():
        candidate = re.sub(r"^\s*(?:[-*]|\d+[.)])\s*", "", line).strip()
        if candidate and candidate.lower() not in {item.lower() for item in variants}:
            variants.append(candidate)
    return variants[: count + 1]


def multi_query_hybrid_children(question: str, k: int = 5) -> list[RetrievedDocument]:
    ranked_lists: list[Sequence[RetrievedDocument]] = []
    for variant in generate_query_variants(question):
        ranked_lists.append(dense_retrieve(variant, k=FIRST_STAGE_K))
        ranked_lists.append(bm25_retrieve(variant, k=FIRST_STAGE_K))
    return reciprocal_rank_fusion(ranked_lists, limit=k)


def multi_query_reranked_retrieve(question: str, k: int = 5) -> list[RetrievedDocument]:
    child_candidates = multi_query_hybrid_children(question, k=RERANK_CANDIDATE_K)
    parent_candidates = recover_parent_documents(child_candidates, k=RERANK_CANDIDATE_K)
    return rerank_parent_candidates(question, parent_candidates, k)


question = "How should an owner prepare for a senior cat wellness visit?"
print(generate_query_variants(question))
print_results(multi_query_reranked_retrieve(question, k=4), text_limit=400)


['How should an owner prepare for a senior cat wellness visit?', 'senior cat wellness visit preparation PDF', 'cat owner prepare for senior cat checkup guideline PDF', 'feline senior wellness exam owner instructions PDF']
#1 | page-06 | page=6 | score=0.7716
For example, some senior cats aged 10 years and older may remain in excellent physical condition and would be best treated as a mature adult at the veterinarian ’s discretion. The guidelines are intended to be a starting point from which individualized care recommenda- tions can be developed. Discussion Items for All Life Stages The Task Force recommends a minimum of annual examinations for all ca

#2 | page-10 | page=10 | score=0.6923
including carpeting, window and door frames, curtains, and couches. Keeping the nails shorter can minimize the damage to household items as well as to people. Moreover, meeting the cat ’s environmental needs may be bene ﬁcial in reducing scratching of unwanted surfaces. 38 Any intercat-related issues

## Task 9: Compare Retrieval and Answer Results

The local library measures retrieval and answer quality separately.

- **Retrieval:** Hit@k, Precision@k, Recall@k, MRR, and latency.
- **Faithfulness:** the share of answer statements supported by the passages retrieved for that answer. The OpenAI judge records a statement, reason, and 0/1 verdict for each claim.
- **Answer similarity:** cosine similarity between OpenAI embeddings of the generated answer and a reviewed reference answer.

First, compare five retrieval pipelines on the same reviewed text-retrieval cases:

1. Naive dense RAG.
2. BM25.
3. Dense parent-child retrieval.
4. Hybrid dense + BM25 retrieval with Cohere reranking.
5. Multi-query hybrid retrieve-then-rerank.

Inspect per-case rankings alongside aggregate metrics before choosing a system. The visual life-stage table needs separate handling: the PDF text extractor loses most of its cells.


In [24]:
text_reviewed_cases = [
    EvalCase(
        id="life-stage-definitions",
        query="What feline life stages does the guideline define?",
        relevant_evidence_ids=("page-03",),
        tags=("baseline", "life-stage"),
    ),
    EvalCase(
        id="senior-visit-frequency",
        query="How often should senior cats be seen by a veterinarian?",
        relevant_evidence_ids=("page-06",),
        tags=("exact", "senior"),
    ),
    EvalCase(
        id="bcs-mcs",
        query="What do BCS and MCS mean, and why are they recorded?",
        relevant_evidence_ids=("page-06", "page-07"),
        tags=("acronym", "dense-vs-bm25"),
    ),
]

table_layout_challenge = EvalCase(
    id="life-stage-table",
    query="How do wellness discussion items change across feline life stages?",
    relevant_evidence_ids=("page-04", "page-05"),
    tags=("table", "parent-child", "requires-visual-review"),
    notes="Use this after adding a PDF-page vision fallback.",
)

EVAL_K = 4
dense_report = run_retrieval_eval("naive_dense", text_reviewed_cases, dense_retrieve, k=EVAL_K)
bm25_report = run_retrieval_eval("bm25", text_reviewed_cases, bm25_retrieve, k=EVAL_K)
parent_child_report = run_retrieval_eval(
    "dense_parent_child",
    text_reviewed_cases,
    parent_child_dense_retrieve,
    k=EVAL_K,
)
hybrid_rerank_report = run_retrieval_eval(
    "hybrid_rrf_then_cohere",
    text_reviewed_cases,
    hybrid_reranked_retrieve,
    k=EVAL_K,
)
multi_query_report = run_retrieval_eval(
    "multi_query_hybrid_rerank",
    text_reviewed_cases,
    multi_query_reranked_retrieve,
    k=EVAL_K,
)

comparison = compare_reports(
    dense_report,
    bm25_report,
    parent_child_report,
    hybrid_rerank_report,
    multi_query_report,
)
comparison


,retriever,k,cases,hit_rate,precision_at_k,recall_at_k,mrr,mean_latency_ms
0,dense_parent_child,4,3,1.0,0.333333,1.0,1.000000,509.400722
1,naive_dense,4,3,1.0,0.333333,1.0,1.000000,946.363084
2,multi_query_hybrid_rerank,4,3,1.0,0.333333,1.0,0.777778,3018.388931
3,hybrid_rrf_then_cohere,4,3,1.0,0.333333,1.0,0.777778,16852.678750
4,bm25,4,3,1.0,0.333333,1.0,0.583333,4.052055


In [31]:
for name, report in [
    ("naive_dense", dense_report),
    ("parent_child", parent_child_report),
    ("bm25", bm25_report),
    ("hybrid", hybrid_rerank_report),
    ("multi_query", multi_query_report),
]:
    print(f"\n=== {name} ===")
    display(report.case_table())


=== naive_dense ===


,case_id,query,tags,retrieved_ids,hit@k,precision@k,recall@k,reciprocal_rank,latency_ms
0,life-stage-definitions,What feline life stages does the guideline def...,"baseline, life-stage","[child-013, child-035, child-007, child-001]",1.0,0.25,1.0,1.0,1826.230459
1,senior-visit-frequency,How often should senior cats be seen by a vete...,"exact, senior","[child-021, child-108, child-034, child-060]",1.0,0.25,1.0,1.0,526.354542
2,bcs-mcs,"What do BCS and MCS mean, and why are they rec...","acronym, dense-vs-bm25","[child-029, child-006, child-030, child-005]",1.0,0.50,1.0,1.0,486.504250



=== parent_child ===


,case_id,query,tags,retrieved_ids,hit@k,precision@k,recall@k,reciprocal_rank,latency_ms
0,life-stage-definitions,What feline life stages does the guideline def...,"baseline, life-stage","[page-03, page-07, page-02, page-01]",1.0,0.25,1.0,1.0,546.484375
1,senior-visit-frequency,How often should senior cats be seen by a vete...,"exact, senior","[page-06, page-16, page-07, page-10]",1.0,0.25,1.0,1.0,358.497916
2,bcs-mcs,"What do BCS and MCS mean, and why are they rec...","acronym, dense-vs-bm25","[page-06, page-01, page-07, page-19]",1.0,0.50,1.0,1.0,623.219875



=== bm25 ===


,case_id,query,tags,retrieved_ids,hit@k,precision@k,recall@k,reciprocal_rank,latency_ms
0,life-stage-definitions,What feline life stages does the guideline def...,"baseline, life-stage","[child-025, child-004, child-074, child-012]",1.0,0.25,1.0,0.25,10.026792
1,senior-visit-frequency,How often should senior cats be seen by a vete...,"exact, senior","[child-021, child-082, child-087, child-072]",1.0,0.25,1.0,1.00,0.803666
2,bcs-mcs,"What do BCS and MCS mean, and why are they rec...","acronym, dense-vs-bm25","[child-082, child-029, child-079, child-030]",1.0,0.50,1.0,0.50,1.325708



=== hybrid ===


,case_id,query,tags,retrieved_ids,hit@k,precision@k,recall@k,reciprocal_rank,latency_ms
0,life-stage-definitions,What feline life stages does the guideline def...,"baseline, life-stage","[page-01, page-02, page-03, page-07]",1.0,0.25,1.0,0.333333,43641.94575
1,senior-visit-frequency,How often should senior cats be seen by a vete...,"exact, senior","[page-06, page-02, page-10, page-16]",1.0,0.25,1.0,1.000000,6162.42125
2,bcs-mcs,"What do BCS and MCS mean, and why are they rec...","acronym, dense-vs-bm25","[page-06, page-13, page-01, page-07]",1.0,0.50,1.0,1.000000,753.66925



=== multi_query ===


,case_id,query,tags,retrieved_ids,hit@k,precision@k,recall@k,reciprocal_rank,latency_ms
0,life-stage-definitions,What feline life stages does the guideline def...,"baseline, life-stage","[page-01, page-02, page-03, page-07]",1.0,0.25,1.0,0.333333,3021.671417
1,senior-visit-frequency,How often should senior cats be seen by a vete...,"exact, senior","[page-06, page-02, page-10, page-16]",1.0,0.25,1.0,1.000000,2851.316750
2,bcs-mcs,"What do BCS and MCS mean, and why are they rec...","acronym, dense-vs-bm25","[page-06, page-13, page-01, page-07]",1.0,0.50,1.0,1.000000,3182.178625


### Answer-Level Evaluation: Faithfulness and Answer Similarity

Retrieval success does not guarantee that an answer stays grounded in its retrieved passages or covers the reviewed answer. The cases below add reviewed reference answers.

The generic runner follows the same shape as Evalite: data, task, and scorers. This comparison uses the naive dense baseline and the complete multi-query pipeline. Each answer is scored against its own retrieved passages for faithfulness, then against the same reviewed reference answer for semantic similarity.


In [25]:
answer_reviewed_cases = [
    EvalItem(
        id="life-stage-definitions",
        input="What feline life stages does the guideline define?",
        expected=(
            "The guidelines define kitten (birth to 1 year), young adult (1 through 6 years), "
            "mature adult (7 through 10 years), and senior (over 10 years). End of life can occur at any age."
        ),
        tags=("life-stage",),
    ),
    EvalItem(
        id="senior-visit-frequency",
        input="How often should senior cats be seen by a veterinarian?",
        expected=(
            "All cats should have at least annual examinations. Senior cats should be seen at least every "
            "6 months, with more frequent visits for chronic conditions."
        ),
        tags=("senior",),
    ),
    EvalItem(
        id="bcs-mcs",
        input="What do BCS and MCS mean, and why are they recorded?",
        expected=(
            "BCS is body condition score and MCS is muscle condition score. Along with body weight, "
            "they should be evaluated and recorded at every life stage to identify changes and trends early."
        ),
        tags=("acronym",),
    ),
]

faithfulness_judge = make_openai_faithfulness_judge(evaluation_model)
answer_scorers = [
    faithfulness_scorer(faithfulness_judge),
    answer_similarity_scorer(embeddings),
]


def answerer_for(retriever):
    return lambda question: answer_with_retriever(question, retriever)


dense_answer_report = run_eval(
    "naive_dense_answer",
    data=answer_reviewed_cases,
    task=answerer_for(dense_retrieve),
    scorers=answer_scorers,
)
full_pipeline_answer_report = run_eval(
    "multi_query_hybrid_rerank_answer",
    data=answer_reviewed_cases,
    task=answerer_for(multi_query_reranked_retrieve),
    scorers=answer_scorers,
)

answer_comparison = compare_eval_reports(
    dense_answer_report,
    full_pipeline_answer_report,
)
answer_comparison

# One report per retrieval pipeline
retrievers = {
    "naive_dense": dense_retrieve,
    "bm25": bm25_retrieve,
    "dense_parent_child": parent_child_dense_retrieve,
    "hybrid_rrf_then_cohere": hybrid_reranked_retrieve,
    "multi_query_hybrid_rerank": multi_query_reranked_retrieve,
}
answer_reports = {}
for name, retriever in retrievers.items():
    print(f"Running answer eval: {name} ...")
    answer_reports[name] = run_eval(
        f"{name}_answer",
        data=answer_reviewed_cases,
        task=answerer_for(retriever),
        scorers=answer_scorers,
    )
# Summary: faithfulness + answer_similarity + latency for ALL pipelines
all_answer_comparison = compare_eval_reports(*answer_reports.values())
all_answer_comparison


Running answer eval: naive_dense ...
Running answer eval: bm25 ...
Running answer eval: dense_parent_child ...
Running answer eval: hybrid_rrf_then_cohere ...
Running answer eval: multi_query_hybrid_rerank ...


,evaluation,cases,faithfulness,answer_similarity,mean_task_latency_ms,mean_scoring_latency_ms
0,dense_parent_child_answer,3,1.000000,0.876268,1463.909778,2898.035639
1,hybrid_rrf_then_cohere_answer,3,1.000000,0.855295,2872.813695,3119.990472
2,naive_dense_answer,3,1.000000,0.844795,1194.685473,1641.359291
3,bm25_answer,3,0.944444,0.750717,998.522583,2376.043014
4,multi_query_hybrid_rerank_answer,3,0.833333,0.858447,3335.225597,2746.536611


In [32]:
rows = []
for name, report in answer_reports.items():
    for row in report.rows:
        rows.append({
            "pipeline": name,
            "case_id": row.item.id,
            "faithfulness": row.score("faithfulness").score,
            "answer_similarity": row.score("answer_similarity").score,
            "task_latency_ms": row.task_latency_ms,
        })
import pandas as pd
pd.DataFrame(rows)

,pipeline,case_id,faithfulness,answer_similarity,task_latency_ms
0,naive_dense,life-stage-definitions,1.000000,0.796580,1466.805875
1,naive_dense,senior-visit-frequency,1.000000,0.835494,1013.238959
2,naive_dense,bcs-mcs,1.000000,0.902311,1104.011584
3,bm25,life-stage-definitions,1.000000,0.562845,961.229666
4,bm25,senior-visit-frequency,1.000000,0.792774,836.206417
5,bm25,bcs-mcs,0.833333,0.896533,1198.131667
6,dense_parent_child,life-stage-definitions,1.000000,0.896472,2038.073125
7,dense_parent_child,senior-visit-frequency,1.000000,0.845941,922.462000
8,dense_parent_child,bcs-mcs,1.000000,0.886392,1431.194209
9,hybrid_rrf_then_cohere,life-stage-definitions,1.000000,0.824169,1332.219042


In [33]:
report = answer_reports["multi_query_hybrid_rerank"]  # or bm25, etc.
row = next(r for r in report.rows if r.item.id == "bcs-mcs")
for v in row.score("faithfulness").metadata["verdicts"]:
    print(v)

{'statement': 'BCS means body condition score, and MCS means muscle condition score.', 'reason': 'Source 3 explicitly expands the abbreviations as "BCS (body condition score)" and "MCS (muscle condition score)."', 'verdict': 1}
{'statement': 'BCS and MCS are recorded because they are important parts of the physical examination at all life stages.', 'reason': 'Source 1 says evaluation and recording of body weight, BCS, and MCS are important components of the physical examination at all life stages.', 'verdict': 1}
{'statement': 'BCS and MCS are recorded to help detect changes early and identify trends over time.', 'reason': 'Source 4 states that obtaining photographs is recommended to facilitate monitoring BCS/MCS as the cat ages and can help detect changes and identify trends. However, it does not directly say BCS/MCS themselves are recorded for that reason; it supports the monitoring rationale only indirectly and partially.', 'verdict': 0}
{'statement': 'Recording BCS and MCS can help

### ❓ Question 4: Is More Retrieval Always Better?

Suppose multi-query plus reranking improves recall but lowers MRR and adds noticeable latency. How would faithfulness and answer similarity change your decision about shipping it for every cat-health question?

If multi query reranking improves recall but lowers MRR and adds noticeable latency, recall alone is not sufficient to ship multi query + reranking. Faithfulness and answer similarity results are needed to assess whether the extra retrieval actually improves user experience with safer better answer - and whether that gain is worth the latency. 

Recall only shows whether we find the relevant evidence in the top k 
MRR shows whether the right evidence is properly ranked affecting grounding risk > this increases faithfulness risk on edge cases  
Faithfulness acts as the safety gate >> if broader retrieval increases unsupported claims, the pipeline is unsuitable for a health app regardless of the recall. Faithfulness drops > default pipeline stays, multiy query is not justified for all questions 
Answer similarity acts as the value gate >> if higher recall does not produce answer closer to reviewed references, the extra latency and API cost is not justified, if the health app is not useful. Even if multi query improves recall on hard cases, if its unable to provide better answers, its not very helpful for the end user 

I would deploy multi query reranking selectively for vague or multi part questions where cheaper/easier pipelines fail faithfulness or similarity checks. As a quick sanity check if faithfulness goes down, should not ship as safety takes precedence and if similarity goes down/flat, should not ship as there is no direct impact to user and is not worth the increased latency/cost





### 🚀 Activity 2: Make and Defend a Retrieval Recommendation

Choose one retrieval result and one answer-evaluation result, then make a recommendation for the cat-health application.

Include:

1. Which rung of the retrieval ladder you chose.
2. Evidence from at least two metrics and one inspected ranking or claim-level verdict.
3. Its cost/latency trade-off.
4. One case where you would choose a different rung instead.

A higher aggregate metric does not settle the product decision.

++++ 

At the aggregate level, all pipelines achieved hit rate and recall of 1.0 (all found the right pages at least once), so retrieval recall alone would not distinguish them. Parent-child and naive dense both reached MRR (how high the right page ranked) 1.0 with mean retrieval latencies of 509 ms and 946 ms, while hybrid and multi-query dropped to MRR 0.78 with mean retrieval latencies of 16.9 s and 3.0 s. At the retrieval aggregate level, parent-child performed the best. 

On the answer side, parent-child achieved the highest mean similarity/answer quality (0.876) and faithfulness 1.0 on all cases; naive dense matched faithfulness but trailed on mean similarity (0.845) but fastest overall (1,195 ms). Multi-query looks decent on mean similarity (0.858) but faithfulness drops to 0.83 on a case by case basis. 

Case-level inspection explains why: 
1. life-stage definitions: parent-child wins because it properly found the right page-03 and has the best similarity score of 0.896. All faithfulness 1.0. hybrid/multi/bms25 buried required page/context under other pages 
2. senior-visit-frequency: simple question and all pipes find the right page first, MRR 1.0, recall 1.0. However parent child was fastest with 358ms, naive_dense 526 ms, hybrid 6,162 ms, multi_query 2,851 ms. In terms of answer all had faithfulness of 1.0, parent_child had highest similarity of 0.846 with hybrid next 0.845 and multi_qury 0.843 third - but parent_child was at 922 ms, hybrid at 5483 ms. heavy pipelines add latency with no significant answer benefit 
3. bcs-mcs: Recall 1.0 for all, BM25 ranked worst with MRR 0.55, rest 1.0. Hybrid and mutli_query same page list, but multi_query slower (3182 ms vs 754 ms). In terms of answer, naive_dense did the best with faithifulness 1.0 and similarity 0.902. Second parent_child with faith 1.0 and similarity 0.886, third hybrid: faith 1.0, sim 0.897. good similarity does not mean good pipeline. multi_query and bm has ok similarity, but faithfulness is not great, especially mult_query at 0.5 (2 claims unsupported, which is pretty bad) 

across cases, key insight is that recall/hit is identical at 1.0 across cases, so MRR and latency are the key drivers for retrieval quality in this example. and in terms of answers, faithfulness 1 for parent_child, naive_dense simiarlity for parent_child best. This shows that if the right pages are ranked first, agents get full context to get to complete answer. Not a faithfulness issue, rather a completeness/context issue. parent_child wins because the proper page-03 leads context, hybrid/multi/bm25 bury it under pages. 

One case where I would choose a different pipe would be for the bcs-mcs case: what do BCS and MCS mean, and why are they recorded? 

Default: dense_parent_child 
Alternate for this case: naive_dense 

naive_dense	
faithfulness 1.0
similarity 0.902
latency: 1,104

dense_parent_child
faithfulness 1.0
similarity 0.886
latency: 1,431

Same safety with faithfulness 1, but better match to reference answer with naive_dense (+0.016), and faster end to end at 1,104. Focused child chunks with the acronym definitions and why recorded texts without extra page noise. In terms of retrieval, both rank the right evidence as shown by rank and recall. Decision comes down to answer quality not retrieval. 

I recommend dense parent-child as the default, naive dense for acronym-heavy questions, and neither hybrid nor multi-query as global defaults.

## Task 10: Answer with a Selected Pipeline

Pass only the selected pipeline's retrieved context to the answer model. Source labels keep the evidence inspectable. For the two-page life-stage table, inspect the original PDF page when text extraction does not preserve the table structure.


In [30]:
result = answer_with_retriever(
    "What should an owner discuss at a senior cat wellness visit?",
    retriever=multi_query_reranked_retrieve,
)
print(result.answer)
print("\nRetrieved evidence:")
print_results(result.documents, text_limit=200)


At a senior cat wellness visit, the owner should discuss:

- Changes in appetite
- Urination and thirst changes, including polyuria and polydipsia
- Vomiting, vomiting hairballs, or diarrhea
- Increased nocturnal activity and vocalization
- Changes in the cat’s normal habits or activity
- Any signs that could suggest cognitive dysfunction, reduced mobility, pain, or reduced vision

The guidelines also note that senior visits should be used to review nutrition, body weight/body condition, and other preventive health topics as part of individualized care.

Sources: page-07, page-06, page-14

Retrieved evidence:
#1 | page-06 | page=6 | score=0.8839
For example, some senior cats aged 10 years and older may remain in excellent physical condition and would be best treated as a mature adult at the veterinarian ’s discretion. The guidelines are inten

#2 | page-07 | page=7 | score=0.8126
detection of changes and identi ﬁcation of trends. 20 Obtaining dorsal and lateral photographs of the patie

## Advanced Builds

- Add maximal marginal relevance (MMR) and measure whether it reduces redundant chunks.
- Add HyDE: generate a hypothetical answer, embed it, and use it as an alternate dense query.
- Add a PDF-page vision fallback for the life-stage table challenge.
- Add a second reviewed corpus and use metadata routing to avoid mixing evidence sets.

## Recap

Build in this order:

1. Start with a transparent naive dense RAG baseline in in-memory Qdrant.
2. Add BM25 to see the value of lexical retrieval.
3. Add parent-child recovery to improve answer context.
4. Combine dense and BM25 with RRF: hybrid / ensemble retrieval.
5. Rerank the hybrid parent candidates with Cohere: a two-stage retrieve-then-rerank pipeline.
6. Add multi-query expansion only when its recall benefit justifies its extra latency and cost.

Use the local evaluation library to inspect the retrieved evidence behind every aggregate number.
